In [34]:
!pip install -q jiwer rapidfuzz

import os
import gc
import ast
import json
import math
import time
import random
import shutil
import warnings
from glob import glob
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm
from jiwer import wer

warnings.filterwarnings("ignore")

print(wer(["hello world"], ["hello world"]))

0.0


In [2]:
# =========================================================
# 1) BASIC CONFIG
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda:0" if USE_CUDA else "cpu")
DEVICE_TYPE = "cuda" if USE_CUDA else "cpu"

print("=" * 80)
print("Device:", DEVICE)
print("CUDA available:", USE_CUDA)
print("GPU count:", torch.cuda.device_count() if USE_CUDA else 0)
print("=" * 80)

# audio / mel
TARGET_SR = 16000
N_MELS = 80
N_FFT = 400
HOP_LENGTH = 160
WIN_LENGTH = 400

# training
BATCH_SIZE = 32
NUM_WORKERS = 0          # مهم: مع RAM preload غالبًا 0 أفضل
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 5.0
PATIENCE = 5
MIN_DELTA = 0.001
EPOCHS_PER_RUN = 8

# model
PROJ_DIM = 384
RNN_HIDDEN = 320
RNN_LAYERS = 2
DROPOUT = 0.20

# augmentation
USE_SPECAUG = True
FREQ_MASK_PARAM = 8
TIME_MASK_PARAM = 20

# bucketing
BUCKET_SIZE_MULTIPLIER = 60

# caching
PRELOAD_TO_RAM = True

# decoding
BLANK_IDX = 0
DEFAULT_BEAM_WIDTH = 20

print("BATCH_SIZE:", BATCH_SIZE)
print("PROJ_DIM:", PROJ_DIM)
print("RNN_HIDDEN:", RNN_HIDDEN)
print("RNN_LAYERS:", RNN_LAYERS)

Device: cuda:0
CUDA available: True
GPU count: 2
BATCH_SIZE: 32
PROJ_DIM: 384
RNN_HIDDEN: 320
RNN_LAYERS: 2


In [3]:
# =========================================================
# 3) AUTO-DISCOVER PATHS
# =========================================================
def find_first(patterns):
    for pattern in patterns:
        matches = glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

INPUT_BASE = "/kaggle/input/datasets/obaidah"
WORK_BASE = "/kaggle/working"

BUNDLE_DIR = find_first([
    "/kaggle/input/datasets/obaidah/asr-full-bundle",
    "/kaggle/input/**/asr-full-bundle",
])

WAV_DIR = None
for candidate in [
    "/kaggle/working/waves",
    "/kaggle/input/datasets/obaidah/asr-nn/waves",
    "/kaggle/input/**/waves",
]:
    found = find_first([candidate]) if "**" in candidate else (candidate if os.path.exists(candidate) else None)
    if found is not None and os.path.exists(found):
        WAV_DIR = found
        break

MEL_CACHE_DIR = os.path.join(WORK_BASE, "mel_cache")
MEL_DATASET_DIR = os.path.join(WORK_BASE, "mel_dataset")
CHECKPOINT_DIR = os.path.join(WORK_BASE, "checkpoints_final_bigru")

os.makedirs(MEL_CACHE_DIR, exist_ok=True)
os.makedirs(MEL_DATASET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_ready_path = os.path.join(BUNDLE_DIR, "train_ready.csv")
val_ready_path   = os.path.join(BUNDLE_DIR, "val_ready.csv")
test_ready_path  = os.path.join(BUNDLE_DIR, "test_ready.csv")
char2idx_path    = os.path.join(BUNDLE_DIR, "char2idx.json")
idx2char_path    = os.path.join(BUNDLE_DIR, "idx2char.json")

print("BUNDLE_DIR   :", BUNDLE_DIR)
print("WAV_DIR      :", WAV_DIR)
print("MEL_CACHE    :", MEL_CACHE_DIR)
print("MEL_DATASET  :", MEL_DATASET_DIR)
print("CHECKPOINTS  :", CHECKPOINT_DIR)

for p in [train_ready_path, val_ready_path, test_ready_path, char2idx_path, idx2char_path]:
    print(os.path.basename(p), "->", os.path.exists(p))

assert BUNDLE_DIR is not None, "Could not find asr-full-bundle"
assert WAV_DIR is not None and os.path.exists(WAV_DIR), "Could not find waves directory"

BUNDLE_DIR   : /kaggle/input/datasets/obaidah/asr-full-bundle
WAV_DIR      : /kaggle/input/datasets/obaidah/asr-nn/waves
MEL_CACHE    : /kaggle/working/mel_cache
MEL_DATASET  : /kaggle/working/mel_dataset
CHECKPOINTS  : /kaggle/working/checkpoints_final_bigru
train_ready.csv -> True
val_ready.csv -> True
test_ready.csv -> True
char2idx.json -> True
idx2char.json -> True


In [5]:
# =========================================================
# 4) LOAD CSV + VOCAB + FIX PATHS
# =========================================================
def parse_label_ids(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return ast.literal_eval(x)

train_df = pd.read_csv(train_ready_path)
val_df   = pd.read_csv(val_ready_path)
test_df  = pd.read_csv(test_ready_path)

for df in [train_df, val_df, test_df]:
    df["label_ids"] = df["label_ids"].apply(parse_label_ids)

with open(char2idx_path, "r", encoding="utf-8") as f:
    char2idx = json.load(f)

with open(idx2char_path, "r", encoding="utf-8") as f:
    idx2char = json.load(f)

idx2char = {int(k): v for k, v in idx2char.items()}

def remap_to_wav_dir(df, wav_dir):
    df = df.copy()
    df["resampled_path"] = df["resampled_path"].apply(
        lambda p: os.path.join(wav_dir, os.path.basename(str(p)))
    )
    return df

train_df = remap_to_wav_dir(train_df, WAV_DIR)
val_df   = remap_to_wav_dir(val_df, WAV_DIR)
test_df  = remap_to_wav_dir(test_df, WAV_DIR)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    exists_mask = df["resampled_path"].apply(os.path.exists)
    print(f"{name}: {exists_mask.sum()}/{len(df)} wav files found")

train_df = train_df[train_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)
val_df   = val_df[val_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)
test_df  = test_df[test_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)
print("Vocab size :", len(char2idx))

Train: 62973/62973 wav files found
Val: 7872/7872 wav files found
Test: 7872/7872 wav files found
Train shape: (62973, 7)
Val shape  : (7872, 7)
Test shape : (7872, 7)
Vocab size : 32


In [6]:
# =========================================================
# 5) OFFLINE MEL CACHE BUILDER
# =========================================================
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
    center=True,
    power=2.0
)

amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

def wav_to_mel_tensor(audio_path):
    waveform, sr = torchaudio.load(audio_path)  # [C, T]

    if waveform.size(0) > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)

    waveform = waveform.squeeze(0)             # [T]
    features = mel_transform(waveform)         # [M, T]
    features = amplitude_to_db(features)       # [M, T]
    features = features.transpose(0, 1)        # [T, M]
    return features.contiguous().float()

def build_feature_path(audio_path, mel_cache_dir):
    base = os.path.splitext(os.path.basename(audio_path))[0]
    return os.path.join(mel_cache_dir, base + ".pt")

def build_mel_cache_for_all_splits(train_df, val_df, test_df, mel_cache_dir):
    all_paths = pd.concat([
        train_df["resampled_path"],
        val_df["resampled_path"],
        test_df["resampled_path"]
    ], axis=0).drop_duplicates().tolist()

    existing = 0
    built = 0
    failed = 0

    for audio_path in tqdm(all_paths, desc="Building mel cache"):
        feature_path = build_feature_path(audio_path, mel_cache_dir)

        if os.path.exists(feature_path):
            existing += 1
            continue

        try:
            feat = wav_to_mel_tensor(audio_path)
            torch.save(feat, feature_path)
            built += 1
        except Exception as e:
            failed += 1
            print(f"Failed: {audio_path} -> {e}")

    print(f"Existing: {existing}")
    print(f"Built   : {built}")
    print(f"Failed  : {failed}")

build_mel_cache_for_all_splits(train_df, val_df, test_df, MEL_CACHE_DIR)

Building mel cache:   0%|          | 0/78717 [00:00<?, ?it/s]

Existing: 0
Built   : 78717
Failed  : 0


In [7]:
# =========================================================
# 6) BUILD MEL-READY CSVs
# =========================================================
def build_mel_ready_df(df, split_name, mel_cache_dir):
    out = df.copy()

    out["feature_path"] = out["resampled_path"].apply(
        lambda p: build_feature_path(p, mel_cache_dir)
    )

    out["feature_exists"] = out["feature_path"].apply(os.path.exists)
    print(f"[{split_name}] matched mel files: {int(out['feature_exists'].sum())}/{len(out)}")

    out = out[out["feature_exists"]].copy().reset_index(drop=True)

    # keep only necessary cols
    out = out[["feature_path", "label_ids", "normalized_text"]].copy()

    # remove empty labels
    out["label_len"] = out["label_ids"].apply(len)
    out = out[out["label_len"] > 0].drop(columns=["label_len"]).reset_index(drop=True)

    return out

train_mel_df = build_mel_ready_df(train_df, "train", MEL_CACHE_DIR)
val_mel_df   = build_mel_ready_df(val_df,   "val",   MEL_CACHE_DIR)
test_mel_df  = build_mel_ready_df(test_df,  "test",  MEL_CACHE_DIR)

train_mel_csv = os.path.join(MEL_DATASET_DIR, "train_mel_ready.csv")
val_mel_csv   = os.path.join(MEL_DATASET_DIR, "val_mel_ready.csv")
test_mel_csv  = os.path.join(MEL_DATASET_DIR, "test_mel_ready.csv")

train_mel_df.to_csv(train_mel_csv, index=False, encoding="utf-8")
val_mel_df.to_csv(val_mel_csv, index=False, encoding="utf-8")
test_mel_df.to_csv(test_mel_csv, index=False, encoding="utf-8")

print(train_mel_csv, os.path.exists(train_mel_csv), train_mel_df.shape)
print(val_mel_csv,   os.path.exists(val_mel_csv),   val_mel_df.shape)
print(test_mel_csv,  os.path.exists(test_mel_csv),  test_mel_df.shape)

[train] matched mel files: 62973/62973
[val] matched mel files: 7872/7872
[test] matched mel files: 7872/7872
/kaggle/working/mel_dataset/train_mel_ready.csv True (62973, 3)
/kaggle/working/mel_dataset/val_mel_ready.csv True (7872, 3)
/kaggle/working/mel_dataset/test_mel_ready.csv True (7872, 3)


In [8]:
# =========================================================
# 7) RAM PRELOAD
# =========================================================
@dataclass
class RAMBundle:
    features: list
    feature_lengths: torch.Tensor
    labels: list
    label_lengths: torch.Tensor
    texts: list

def preload_features_to_ram(df, name="split"):
    features = []
    feature_lengths = []
    labels = []
    label_lengths = []
    texts = []

    start = time.time()

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Preloading {name}"):
        x = torch.load(row["feature_path"], map_location="cpu").float().contiguous()
        y = torch.tensor(row["label_ids"], dtype=torch.long)

        features.append(x)
        feature_lengths.append(x.size(0))
        labels.append(y)
        label_lengths.append(len(y))
        texts.append(row["normalized_text"])

    feature_lengths = torch.tensor(feature_lengths, dtype=torch.long)
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)

    print(f"{name} loaded in {(time.time() - start)/60:.2f} min")
    print(f"{name} samples: {len(features)}")
    print(f"{name} avg T   : {feature_lengths.float().mean().item():.2f}")
    print(f"{name} max T   : {feature_lengths.max().item()}")

    return RAMBundle(
        features=features,
        feature_lengths=feature_lengths,
        labels=labels,
        label_lengths=label_lengths,
        texts=texts
    )

if PRELOAD_TO_RAM:
    train_bundle = preload_features_to_ram(train_mel_df, "train")
    val_bundle   = preload_features_to_ram(val_mel_df,   "val")
    test_bundle  = preload_features_to_ram(test_mel_df,  "test")
else:
    train_bundle = val_bundle = test_bundle = None

Preloading train:   0%|          | 0/62973 [00:00<?, ?it/s]

train loaded in 1.50 min
train samples: 62973
train avg T   : 408.79
train max T   : 1059


Preloading val:   0%|          | 0/7872 [00:00<?, ?it/s]

val loaded in 0.18 min
val samples: 7872
val avg T   : 408.05
val max T   : 1027


Preloading test:   0%|          | 0/7872 [00:00<?, ?it/s]

test loaded in 0.18 min
test samples: 7872
test avg T   : 407.40
test max T   : 1088


In [10]:
# =========================================================
# 8) FAST DATASET + BUCKETING
# =========================================================
class RAMMelASRDataset(Dataset):
    def __init__(self, bundle, use_specaug=False, freq_mask_param=8, time_mask_param=20):
        self.features = bundle.features
        self.feature_lengths = bundle.feature_lengths
        self.labels = bundle.labels
        self.label_lengths = bundle.label_lengths
        self.texts = bundle.texts

        self.use_specaug = use_specaug
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def __len__(self):
        return len(self.features)

    def apply_specaug(self, features):
        x = features.transpose(0, 1).contiguous()  # [M, T]

        if self.freq_mask_param > 0 and x.size(0) > 1:
            f = random.randint(0, min(self.freq_mask_param, x.size(0) - 1))
            if f > 0:
                f0 = random.randint(0, x.size(0) - f)
                x[f0:f0 + f, :] = 0

        if self.time_mask_param > 0 and x.size(1) > 1:
            t = random.randint(0, min(self.time_mask_param, x.size(1) - 1))
            if t > 0:
                t0 = random.randint(0, x.size(1) - t)
                x[:, t0:t0 + t] = 0

        return x.transpose(0, 1).contiguous()

    def __getitem__(self, idx):
        feat = self.features[idx]
        lab = self.labels[idx]

        if self.use_specaug:
            feat = self.apply_specaug(feat.clone())

        return {
            "features": feat,
            "feature_length": int(self.feature_lengths[idx]),
            "labels": lab,
            "label_length": int(self.label_lengths[idx]),
            "text": self.texts[idx]
        }

class BucketBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, shuffle=True, bucket_size_multiplier=60):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.bucket_size = batch_size * bucket_size_multiplier
        self.indices = np.arange(len(self.lengths))

    def __iter__(self):
        indices = self.indices.copy()

        if self.shuffle:
            np.random.shuffle(indices)

        buckets = []
        for i in range(0, len(indices), self.bucket_size):
            bucket = indices[i:i+self.bucket_size]
            bucket = sorted(bucket, key=lambda x: self.lengths[x])
            buckets.append(bucket)

        batches = []
        for bucket in buckets:
            for i in range(0, len(bucket), self.batch_size):
                batch = bucket[i:i+self.batch_size]
                if len(batch) > 0:
                    batches.append(batch)

        if self.shuffle:
            np.random.shuffle(batches)

        for batch in batches:
            yield batch

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

def collate_fn_fast(batch):
    if len(batch) == 0:
        return None

    features = [b["features"] for b in batch]
    labels = [b["labels"] for b in batch]

    feature_lengths = torch.tensor([b["feature_length"] for b in batch], dtype=torch.long)
    label_lengths = torch.tensor([b["label_length"] for b in batch], dtype=torch.long)

    B = len(features)
    T = int(feature_lengths.max().item())
    M = int(features[0].size(1))

    padded = torch.zeros(B, T, M, dtype=features[0].dtype)
    for i, feat in enumerate(features):
        padded[i, :feat.size(0)] = feat

    return {
        "features": padded,
        "feature_lengths": feature_lengths,   # CPU
        "labels": torch.cat(labels, dim=0),
        "label_lengths": label_lengths,       # CPU
        "texts": [b["text"] for b in batch]
    }

train_dataset = RAMMelASRDataset(
    train_bundle,
    use_specaug=USE_SPECAUG,
    freq_mask_param=FREQ_MASK_PARAM,
    time_mask_param=TIME_MASK_PARAM
)

val_dataset = RAMMelASRDataset(
    val_bundle,
    use_specaug=False
)

test_dataset = RAMMelASRDataset(
    test_bundle,
    use_specaug=False
)

train_batch_sampler = BucketBatchSampler(
    lengths=train_bundle.feature_lengths.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=True,
    bucket_size_multiplier=BUCKET_SIZE_MULTIPLIER
)

val_batch_sampler = BucketBatchSampler(
    lengths=val_bundle.feature_lengths.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=False,
    bucket_size_multiplier=BUCKET_SIZE_MULTIPLIER
)

test_batch_sampler = BucketBatchSampler(
    lengths=test_bundle.feature_lengths.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=False,
    bucket_size_multiplier=BUCKET_SIZE_MULTIPLIER
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn_fast
)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_batch_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn_fast
)

test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_batch_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn_fast
)

sample_batch = next(iter(train_loader))
print("features shape       :", sample_batch["features"].shape)
print("feature_lengths shape:", sample_batch["feature_lengths"].shape)
print("labels shape         :", sample_batch["labels"].shape)
print("label_lengths shape  :", sample_batch["label_lengths"].shape)
print("sample text          :", sample_batch["texts"][0])

features shape       : torch.Size([32, 515, 80])
feature_lengths shape: torch.Size([32])
labels shape         : torch.Size([1173])
label_lengths shape  : torch.Size([32])
sample text          : هل انت تفكر جديا في شراء هذه السيارة القديمة


In [18]:
# =========================================================
# 1) SETUP + CONFIG
# =========================================================
!pip -q install jiwer

import os
import ast
import json
import time
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# For speed
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda:0" if USE_CUDA else "cpu")
DEVICE_TYPE = "cuda" if USE_CUDA else "cpu"

print("=" * 70)
print("Device:", DEVICE)
print("CUDA available:", USE_CUDA)
print("GPU count:", torch.cuda.device_count() if USE_CUDA else 0)
print("=" * 70)


# =========================================================
# PATHS (FINAL CORRECT VERSION)
# =========================================================
import os

DATA_DIR = "/kaggle/working/mel_dataset"
CHECKPOINT_DIR = "/kaggle/working/checkpoints_ram_fast_v1"
INPUT_BUNDLE_DIR = "/kaggle/input/datasets/obaidah/asr-full-bundle"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_mel_csv = os.path.join(DATA_DIR, "train_mel_ready.csv")
val_mel_csv   = os.path.join(DATA_DIR, "val_mel_ready.csv")
test_mel_csv  = os.path.join(DATA_DIR, "test_mel_ready.csv")

char2idx_path = os.path.join(INPUT_BUNDLE_DIR, "char2idx.json")
idx2char_path = os.path.join(INPUT_BUNDLE_DIR, "idx2char.json")

last_ckpt_path = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
history_path = os.path.join(CHECKPOINT_DIR, "history.json")
training_state_path = os.path.join(CHECKPOINT_DIR, "training_state.json")

print("=" * 70)
print("train_mel_csv exists:", os.path.exists(train_mel_csv))
print("val_mel_csv exists  :", os.path.exists(val_mel_csv))
print("test_mel_csv exists :", os.path.exists(test_mel_csv))
print("char2idx exists     :", os.path.exists(char2idx_path))
print("idx2char exists     :", os.path.exists(idx2char_path))
print("Checkpoint dir      :", CHECKPOINT_DIR)
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.1 MB/s eta 0:00:00:00:01
Device: cuda:0
CUDA available: True
GPU count: 2
train_mel_csv exists: True
val_mel_csv exists  : True
test_mel_csv exists : True
char2idx exists     : True
idx2char exists     : True
Checkpoint dir      : /kaggle/working/checkpoints_ram_fast_v1


In [11]:
# =========================================================
# 9) MODEL: CNN + BiGRU + CTC
# =========================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.block(x)

class CNNBiGRUCTC(nn.Module):
    def __init__(
        self,
        num_classes,
        n_mels=80,
        proj_dim=384,
        rnn_hidden=320,
        rnn_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.cnn1 = ConvBlock(1, 32, dropout=0.10)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.cnn2 = ConvBlock(32, 64, dropout=0.10)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.cnn3 = ConvBlock(64, 128, dropout=dropout)
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))

        # mel: 80 -> 40 -> 20 -> 20
        cnn_out_dim = 128 * 20

        self.projection = nn.Sequential(
            nn.Linear(cnn_out_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.bigru = nn.GRU(
            input_size=proj_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            dropout=dropout if rnn_layers > 1 else 0.0,
            bidirectional=True
        )

        self.classifier = nn.Linear(rnn_hidden * 2, num_classes)

    def downsample_lengths(self, input_lengths_cpu):
        output_lengths = input_lengths_cpu.clone()
        output_lengths = torch.div(output_lengths, 2, rounding_mode="floor")
        output_lengths = torch.div(output_lengths, 2, rounding_mode="floor")
        output_lengths = torch.div(output_lengths, 2, rounding_mode="floor")
        output_lengths = torch.clamp(output_lengths, min=1)
        return output_lengths

    def forward(self, x, input_lengths_cpu):
        # x: [B, T, 80]
        x = x.unsqueeze(1)         # [B, 1, T, 80]

        x = self.cnn1(x)
        x = self.pool1(x)

        x = self.cnn2(x)
        x = self.pool2(x)

        x = self.cnn3(x)
        x = self.pool3(x)          # [B, 128, T/8, 20]

        b, c, t, f = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(b, t, c * f)    # [B, T', 2560]

        x = self.projection(x)     # [B, T', proj_dim]

        output_lengths_cpu = self.downsample_lengths(input_lengths_cpu)

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths=output_lengths_cpu,
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.bigru(packed)

        x, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        logits = self.classifier(x)                # [B, T', C]
        log_probs = F.log_softmax(logits, dim=-1)  # [B, T', C]

        return log_probs, output_lengths_cpu

num_classes = len(char2idx)

model = CNNBiGRUCTC(
    num_classes=num_classes,
    n_mels=N_MELS,
    proj_dim=PROJ_DIM,
    rnn_hidden=RNN_HIDDEN,
    rnn_layers=RNN_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

CNNBiGRUCTC(
  (cnn1): ConvBlock(
    (block): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Dropout(p=0.1, inplace=False)
    )
  )
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (cnn2): ConvBlock(
    (block): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=T

In [40]:
# =========================================================
# 10) TRAINING CONFIG + CHECKPOINTING
# =========================================================
last_ckpt_path = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
history_path = os.path.join(CHECKPOINT_DIR, "history.json")
training_state_path = os.path.join(CHECKPOINT_DIR, "training_state.json")

criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    threshold=MIN_DELTA,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler(DEVICE_TYPE, enabled=USE_CUDA)

best_val_loss = float("inf")
start_epoch = 0
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "val_wer_greedy": []
}

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def save_checkpoint(path, epoch, best_val_loss, epochs_without_improvement, history):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
        "config": {
            "BATCH_SIZE": BATCH_SIZE,
            "PROJ_DIM": PROJ_DIM,
            "RNN_HIDDEN": RNN_HIDDEN,
            "RNN_LAYERS": RNN_LAYERS,
            "DROPOUT": DROPOUT
        }
    }, path)

def load_checkpoint_if_exists(path):
    global best_val_loss, start_epoch, epochs_without_improvement, history

    if os.path.exists(path):
        print(f"\nFound checkpoint: {path}")
        ckpt = torch.load(path, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

        if ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = ckpt.get("epoch", -1) + 1
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        epochs_without_improvement = ckpt.get("epochs_without_improvement", 0)
        history = ckpt.get("history", history)

        print("Resumed from epoch:", start_epoch)
        print("Best val loss so far:", best_val_loss)
        print("epochs_without_improvement:", epochs_without_improvement)
    else:
        print("\nNo checkpoint found. Starting from scratch.")

# لو عايز تكمل من checkpoint سابق من نفس الموديل الجديد
load_checkpoint_if_exists(last_ckpt_path)


Found checkpoint: /kaggle/working/checkpoints_final_bigru/last_checkpoint.pt
Resumed from epoch: 24
Best val loss so far: 0.6986117882699501
epochs_without_improvement: 1


In [13]:
# =========================================================
# 11) GREEDY DECODE + WER
# =========================================================
def ids_to_text(token_ids, idx2char):
    chars = []
    for idx in token_ids:
        idx = int(idx)
        if idx in idx2char:
            chars.append(idx2char[idx])
    return "".join(chars).strip()

def ctc_greedy_decode(log_probs, output_lengths, blank_idx=0):
    pred_ids = torch.argmax(log_probs, dim=-1)  # [B, T]
    decoded_batch = []

    for b in range(pred_ids.size(0)):
        seq_len = int(output_lengths[b].item())
        seq = pred_ids[b][:seq_len].tolist()

        decoded = []
        prev = None
        for token in seq:
            if token != blank_idx and token != prev:
                decoded.append(token)
            prev = token

        decoded_batch.append(decoded)

    return decoded_batch

@torch.no_grad()
def evaluate_wer_greedy(model, loader, idx2char, device, max_batches=None, show_examples=5):
    model.eval()

    references = []
    hypotheses = []

    pbar = tqdm(loader, desc="Greedy WER", leave=False)

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        if max_batches is not None and batch_idx >= max_batches:
            break

        features = batch["features"].to(device, non_blocking=True)
        feature_lengths_cpu = batch["feature_lengths"]
        texts = batch["texts"]

        with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
            log_probs, output_lengths = model(features, feature_lengths_cpu)

        decoded_ids = ctc_greedy_decode(log_probs, output_lengths, blank_idx=BLANK_IDX)
        decoded_texts = [ids_to_text(ids, idx2char) for ids in decoded_ids]

        references.extend(texts)
        hypotheses.extend(decoded_texts)

        current_wer = wer(references, hypotheses)
        pbar.set_postfix({"WER": f"{current_wer:.4f}"})

    final_wer = wer(references, hypotheses)

    print("=" * 80)
    print(f"Final Validation WER (Greedy): {final_wer:.4f}")
    print("=" * 80)

    n = min(show_examples, len(references))
    print("\nSample predictions:\n")
    for i in range(n):
        print(f"[{i+1}] REF : {references[i]}")
        print(f"    HYP : {hypotheses[i]}")
        print("-" * 80)

    return final_wer, references, hypotheses

In [14]:
# =========================================================
# 12) OPTIONAL: BEAM SEARCH + SIMPLE N-GRAM RESCORING
# =========================================================
def logsumexp_pair(a, b):
    if a == -float("inf"):
        return b
    if b == -float("inf"):
        return a
    m = max(a, b)
    return m + math.log(math.exp(a - m) + math.exp(b - m))

def ctc_prefix_beam_search_single(log_probs, beam_width=10, blank_idx=0):
    T, C = log_probs.shape
    beam = {(): (0.0, -float("inf"))}

    for t in range(T):
        next_beam = defaultdict(lambda: (-float("inf"), -float("inf")))
        step = log_probs[t]
        topk = min(beam_width, C)
        top_tokens = torch.topk(step, k=topk).indices.tolist()

        for prefix, (p_b, p_nb) in beam.items():
            for c in top_tokens:
                p = step[c].item()

                if c == blank_idx:
                    nb_p_b, nb_p_nb = next_beam[prefix]
                    nb_p_b = logsumexp_pair(nb_p_b, p_b + p)
                    nb_p_b = logsumexp_pair(nb_p_b, p_nb + p)
                    next_beam[prefix] = (nb_p_b, nb_p_nb)
                else:
                    end_t = prefix[-1] if len(prefix) > 0 else None
                    new_prefix = prefix + (c,)

                    if c == end_t:
                        nb_p_b, nb_p_nb = next_beam[new_prefix]
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_b + p)
                        next_beam[new_prefix] = (nb_p_b, nb_p_nb)

                        nb_p_b2, nb_p_nb2 = next_beam[prefix]
                        nb_p_nb2 = logsumexp_pair(nb_p_nb2, p_nb + p)
                        next_beam[prefix] = (nb_p_b2, nb_p_nb2)
                    else:
                        nb_p_b, nb_p_nb = next_beam[new_prefix]
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_b + p)
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_nb + p)
                        next_beam[new_prefix] = (nb_p_b, nb_p_nb)

        beam = dict(
            sorted(
                next_beam.items(),
                key=lambda x: logsumexp_pair(x[1][0], x[1][1]),
                reverse=True
            )[:beam_width]
        )

    best_prefix = max(beam.items(), key=lambda x: logsumexp_pair(x[1][0], x[1][1]))[0]
    return list(best_prefix)

def ctc_beam_decode_batch(log_probs, output_lengths, beam_width=10, blank_idx=0):
    decoded_batch = []
    for b in range(log_probs.size(0)):
        seq_len = int(output_lengths[b].item())
        lp = log_probs[b, :seq_len].detach().cpu()
        decoded_ids = ctc_prefix_beam_search_single(
            lp,
            beam_width=beam_width,
            blank_idx=blank_idx
        )
        decoded_batch.append(decoded_ids)
    return decoded_batch

@torch.no_grad()
def evaluate_wer_beam(model, loader, idx2char, device, beam_width=10, max_batches=None, show_examples=5):
    model.eval()

    references = []
    hypotheses = []

    pbar = tqdm(loader, desc=f"Beam WER (beam={beam_width})", leave=False)

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        if max_batches is not None and batch_idx >= max_batches:
            break

        features = batch["features"].to(device, non_blocking=True)
        feature_lengths_cpu = batch["feature_lengths"]
        texts = batch["texts"]

        with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
            log_probs, output_lengths = model(features, feature_lengths_cpu)

        decoded_ids = ctc_beam_decode_batch(
            log_probs=log_probs,
            output_lengths=output_lengths,
            beam_width=beam_width,
            blank_idx=BLANK_IDX
        )
        decoded_texts = [ids_to_text(ids, idx2char) for ids in decoded_ids]

        references.extend(texts)
        hypotheses.extend(decoded_texts)

        current_wer = wer(references, hypotheses)
        pbar.set_postfix({"WER": f"{current_wer:.4f}"})

    final_wer = wer(references, hypotheses)

    print("=" * 80)
    print(f"Final Validation WER (Beam Search, beam_width={beam_width}): {final_wer:.4f}")
    print("=" * 80)

    n = min(show_examples, len(references))
    for i in range(n):
        print(f"[{i+1}] REF : {references[i]}")
        print(f"    HYP : {hypotheses[i]}")
        print("-" * 80)

    return final_wer, references, hypotheses

In [19]:
# =========================================================
# 13) OPTIONAL: Manual Beam + Simple N-gram Rescoring
# =========================================================

def normalize_ar_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    text = " ".join(text.split())
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    return text


class SimpleWordNGram:
    def __init__(self, n=3, alpha=0.1):
        self.n = n
        self.alpha = alpha
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()
        self.vocab_size = 0

    def fit(self, texts):
        for text in texts:
            tokens = normalize_ar_text(text).split()
            if not tokens:
                continue

            self.vocab.update(tokens)
            padded = ["<s>"] * (self.n - 1) + tokens + ["</s>"]

            for i in range(len(padded) - self.n + 1):
                ngram = tuple(padded[i:i+self.n])
                context = tuple(padded[i:i+self.n-1])
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

        self.vocab_size = len(self.vocab) + 1

    def log_prob(self, text):
        tokens = normalize_ar_text(text).split()
        if not tokens:
            return -50.0

        padded = ["<s>"] * (self.n - 1) + tokens + ["</s>"]
        total = 0.0

        for i in range(len(padded) - self.n + 1):
            ngram = tuple(padded[i:i+self.n])
            context = tuple(padded[i:i+self.n-1])

            ngram_count = self.ngram_counts[ngram]
            context_count = self.context_counts[context]

            prob = (ngram_count + self.alpha) / (context_count + self.alpha * self.vocab_size)
            total += math.log(prob)

        return total


def logsumexp_pair(a, b):
    if a == -float("inf"):
        return b
    if b == -float("inf"):
        return a
    m = max(a, b)
    return m + math.log(math.exp(a - m) + math.exp(b - m))


def ctc_prefix_beam_search_nbest_single(log_probs, beam_width=20, blank_idx=0):
    """
    log_probs: [T, C] on CPU
    returns: list of tuples (token_ids, acoustic_score)
    """
    T, C = log_probs.shape
    beam = {(): (0.0, -float("inf"))}  # prefix -> (p_blank, p_non_blank)

    for t in range(T):
        next_beam = defaultdict(lambda: (-float("inf"), -float("inf")))
        step = log_probs[t]
        topk = min(beam_width, C)
        top_tokens = torch.topk(step, k=topk).indices.tolist()

        for prefix, (p_b, p_nb) in beam.items():
            for c in top_tokens:
                p = step[c].item()

                if c == blank_idx:
                    nb_p_b, nb_p_nb = next_beam[prefix]
                    nb_p_b = logsumexp_pair(nb_p_b, p_b + p)
                    nb_p_b = logsumexp_pair(nb_p_b, p_nb + p)
                    next_beam[prefix] = (nb_p_b, nb_p_nb)
                else:
                    end_t = prefix[-1] if len(prefix) > 0 else None
                    new_prefix = prefix + (c,)

                    if c == end_t:
                        nb_p_b, nb_p_nb = next_beam[new_prefix]
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_b + p)
                        next_beam[new_prefix] = (nb_p_b, nb_p_nb)

                        nb_p_b2, nb_p_nb2 = next_beam[prefix]
                        nb_p_nb2 = logsumexp_pair(nb_p_nb2, p_nb + p)
                        next_beam[prefix] = (nb_p_b2, nb_p_nb2)
                    else:
                        nb_p_b, nb_p_nb = next_beam[new_prefix]
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_b + p)
                        nb_p_nb = logsumexp_pair(nb_p_nb, p_nb + p)
                        next_beam[new_prefix] = (nb_p_b, nb_p_nb)

        beam = dict(
            sorted(
                next_beam.items(),
                key=lambda x: logsumexp_pair(x[1][0], x[1][1]),
                reverse=True
            )[:beam_width]
        )

    nbest = []
    for prefix, (p_b, p_nb) in beam.items():
        score = logsumexp_pair(p_b, p_nb)
        nbest.append((list(prefix), score))

    nbest = sorted(nbest, key=lambda x: x[1], reverse=True)
    return nbest


lm = SimpleWordNGram(n=3, alpha=0.1)
lm.fit(train_mel_df["normalized_text"].astype(str).tolist())


def decode_batch_with_ngram_rescoring(log_probs, output_lengths, idx2char, beam_width=20, lm_weight=0.6, word_score=0.0):
    decoded_texts = []

    for b in range(log_probs.size(0)):
        seq_len = int(output_lengths[b].item())
        lp = log_probs[b, :seq_len].detach().cpu()

        nbest = ctc_prefix_beam_search_nbest_single(
            lp,
            beam_width=beam_width,
            blank_idx=BLANK_IDX
        )

        rescored = []
        for token_ids, acoustic_score in nbest:
            text = normalize_ar_text(ids_to_text(token_ids, idx2char))
            lm_score = lm.log_prob(text)
            total_score = acoustic_score + lm_weight * lm_score + word_score * len(text.split())
            rescored.append((text, total_score))

        rescored = sorted(rescored, key=lambda x: x[1], reverse=True)
        best_text = rescored[0][0] if len(rescored) > 0 else ""
        decoded_texts.append(best_text)

    return decoded_texts


@torch.no_grad()
def evaluate_wer_ngram_rescoring(
    model,
    loader,
    device,
    idx2char,
    beam_width=20,
    lm_weight=0.6,
    word_score=0.0,
    max_batches=None,
    show_examples=5
):
    model.eval()

    references = []
    hypotheses = []

    pbar = tqdm(loader, desc=f"NGram Rescore WER (beam={beam_width}, lm_w={lm_weight})", leave=False)

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        if max_batches is not None and batch_idx >= max_batches:
            break

        features = batch["features"].to(device, non_blocking=True)
        feature_lengths_cpu = batch["feature_lengths"]
        texts = batch["texts"]

        with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
            log_probs, output_lengths = model(features, feature_lengths_cpu)

        decoded_texts = decode_batch_with_ngram_rescoring(
            log_probs=log_probs,
            output_lengths=output_lengths,
            idx2char=idx2char,
            beam_width=beam_width,
            lm_weight=lm_weight,
            word_score=word_score
        )

        texts = [normalize_ar_text(t) for t in texts]
        decoded_texts = [normalize_ar_text(t) for t in decoded_texts]

        references.extend(texts)
        hypotheses.extend(decoded_texts)

        current_wer = wer(references, hypotheses)
        pbar.set_postfix({"WER": f"{current_wer:.4f}"})

    final_wer = wer(references, hypotheses)

    print("=" * 90)
    print(f"Final Validation WER (Manual Beam + N-gram Rescoring): {final_wer:.4f}")
    print("=" * 90)

    n = min(show_examples, len(references))
    for i in range(n):
        print(f"[{i+1}] REF : {references[i]}")
        print(f"    HYP : {hypotheses[i]}")
        print("-" * 90)

    return final_wer, references, hypotheses

In [20]:
# =========================================================
# 14) TRAIN / VALIDATE LOOPS
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for batch in pbar:
        if batch is None:
            continue

        features = batch["features"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        # keep lengths on CPU
        feature_lengths_cpu = batch["feature_lengths"]
        label_lengths_cpu = batch["label_lengths"]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
            log_probs, output_lengths_cpu = model(features, feature_lengths_cpu)
            log_probs = log_probs.permute(1, 0, 2)  # [T, B, C]

            loss = criterion(
                log_probs,
                labels,
                output_lengths_cpu,
                label_lengths_cpu
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        processed_batches += 1

        avg_loss = running_loss / processed_batches
        pbar.set_postfix({
            "train_loss": f"{avg_loss:.4f}",
            "batch_loss": f"{loss.item():.4f}"
        })

    if processed_batches == 0:
        return float("inf")

    return running_loss / processed_batches


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    for batch in pbar:
        if batch is None:
            continue

        features = batch["features"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        feature_lengths_cpu = batch["feature_lengths"]
        label_lengths_cpu = batch["label_lengths"]

        with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
            log_probs, output_lengths_cpu = model(features, feature_lengths_cpu)
            log_probs = log_probs.permute(1, 0, 2)

            loss = criterion(
                log_probs,
                labels,
                output_lengths_cpu,
                label_lengths_cpu
            )

        running_loss += loss.item()
        processed_batches += 1

        avg_loss = running_loss / processed_batches
        pbar.set_postfix({
            "val_loss": f"{avg_loss:.4f}",
            "batch_loss": f"{loss.item():.4f}"
        })

    if processed_batches == 0:
        return float("inf")

    return running_loss / processed_batches

In [41]:
# =========================================================
# 15) MAIN TRAIN LOOP
# =========================================================
print("\nStarting training ...")
end_epoch = start_epoch + EPOCHS_PER_RUN
print(f"Will train from epoch {start_epoch} to {end_epoch - 1}")

best_val_wer = float("inf")

for epoch in range(start_epoch, end_epoch):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch + 1}")
    print("=" * 80)

    epoch_start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
    val_loss = validate_one_epoch(model, val_loader, criterion, DEVICE)

    # تقييم greedy كل epoch
    val_wer_greedy, _, _ = evaluate_wer_greedy(
        model=model,
        loader=val_loader,
        idx2char=idx2char,
        device=DEVICE,
        max_batches=50,     # أثناء التدريب خليه subset للسرعة
        show_examples=2
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_wer_greedy"].append(val_wer_greedy)

    epoch_time_min = (time.time() - epoch_start_time) / 60.0
    improvement = best_val_loss - val_loss if best_val_loss != float("inf") else 0.0
    generalization_gap = val_loss - train_loss

    print(f"Epoch {epoch + 1}:")
    print(f"  train_loss         = {train_loss:.4f}")
    print(f"  val_loss           = {val_loss:.4f}")
    print(f"  val_wer_greedy     = {val_wer_greedy:.4f}")
    print(f"  improvement(loss)  = {improvement:.6f}")
    print(f"  generalization_gap = {generalization_gap:.4f}")
    print(f"  time               = {epoch_time_min:.2f} min")

    # always save last
    save_checkpoint(
        path=last_ckpt_path,
        epoch=epoch,
        best_val_loss=best_val_loss,
        epochs_without_improvement=epochs_without_improvement,
        history=history
    )

    save_json(history_path, history)
    save_json(training_state_path, {
        "last_completed_epoch": epoch,
        "next_start_epoch": epoch + 1,
        "best_val_loss": best_val_loss,
        "best_val_wer": best_val_wer,
        "epochs_without_improvement": epochs_without_improvement,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "epochs_per_run": EPOCHS_PER_RUN,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "last_train_loss": train_loss,
        "last_val_loss": val_loss,
        "last_val_wer_greedy": val_wer_greedy,
        "last_improvement": improvement,
        "last_generalization_gap": generalization_gap,
        "last_epoch_time_min": epoch_time_min
    })

    improved = val_loss < (best_val_loss - MIN_DELTA)

    if improved:
        best_val_loss = val_loss
        best_val_wer = min(best_val_wer, val_wer_greedy)
        epochs_without_improvement = 0

        save_checkpoint(
            path=best_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        print(f"Saved BEST model at epoch {epoch + 1} with val_loss = {val_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"No significant improvement. Counter: {epochs_without_improvement}/{PATIENCE}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}")
        print(f"Best val_loss = {best_val_loss:.4f}")
        break

print("\nTraining session complete.")
print("Saved files:")
print(" -", last_ckpt_path)
print(" -", best_ckpt_path)
print(" -", history_path)
print(" -", training_state_path)


Starting training ...
Will train from epoch 24 to 31

Epoch 25


Training:   0%|          | 0/1968 [00:00<?, ?it/s]

Validation:   0%|          | 0/246 [00:00<?, ?it/s]

Greedy WER:   0%|          | 0/246 [00:00<?, ?it/s]

Final Validation WER (Greedy): 0.5421

Sample predictions:

[1] REF : من وظفك
    HYP : من ودفك
--------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
--------------------------------------------------------------------------------
Epoch 25:
  train_loss         = 0.4478
  val_loss           = 0.6984
  val_wer_greedy     = 0.5421
  improvement(loss)  = 0.000254
  generalization_gap = 0.2506
  time               = 2.69 min
No significant improvement. Counter: 2/5

Epoch 26


...

Final Validation WER (Greedy): 0.5270

Sample predictions:

[1] REF : من وظفك
    HYP : من وظفك
--------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
--------------------------------------------------------------------------------
Epoch 31:
  train_loss         = 0.3848
  val_loss           = 0.6939
  val_wer_greedy     = 0.5270
  improvement(loss)  = -0.006263
  generalization_gap = 0.3090
  time               = 2.88 min
No significant improvement. Counter: 1/5

Epoch 32


Training:   0%|          | 0/1968 [00:00<?, ?it/s]

Validation:   0%|          | 0/246 [00:00<?, ?it/s]

Greedy WER:   0%|          | 0/246 [00:00<?, ?it/s]

Final Validation WER (Greedy): 0.5238

Sample predictions:

[1] REF : من وظفك
    HYP : من وظفك
--------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
--------------------------------------------------------------------------------
Epoch 32:
  train_loss         = 0.3826
  val_loss           = 0.6937
  val_wer_greedy     = 0.5238
  improvement(loss)  = -0.006083
  generalization_gap = 0.3111
  time               = 2.68 min
No significant improvement. Counter: 2/5

Training session complete.
Saved files:
 - /kaggle/working/checkpoints_final_bigru/last_checkpoint.pt
 - /kaggle/working/checkpoints_final_bigru/best_model.pt
 - /kaggle/working/checkpoints_final_bigru/history.json
 - /kaggle/working/checkpoints_final_bigru/training_state.json


In [44]:
# =========================================================
# 16) FINAL EVALUATION
# =========================================================
# Load best model before final eval
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])
    print("Loaded BEST checkpoint for final evaluation.")

greedy_wer, greedy_refs, greedy_hyps = evaluate_wer_greedy(
    model=model,
    loader=val_loader,
    idx2char=idx2char,
    device=DEVICE,
    max_batches=None,
    show_examples=10
)

beam_wer, beam_refs, beam_hyps = evaluate_wer_beam(
    model=model,
    loader=val_loader,
    idx2char=idx2char,
    device=DEVICE,
    beam_width=DEFAULT_BEAM_WIDTH,
    max_batches=None,
    show_examples=10
)

ngram_wer, ngram_refs, ngram_hyps = evaluate_wer_ngram_rescoring(
    model=model,
    loader=val_loader,
    device=DEVICE,
    idx2char=idx2char,     
    beam_width=DEFAULT_BEAM_WIDTH,
    lm_weight=0.6,
    word_score=0.0,
    max_batches=None,
    show_examples=10
)

print("\nFinal Summary")
print("=" * 80)
print(f"Greedy WER            : {greedy_wer:.4f}")
print(f"Beam Search WER       : {beam_wer:.4f}")
print(f"N-gram Rescore WER    : {ngram_wer:.4f}")
print("=" * 80)

Loaded BEST checkpoint for final evaluation.


Greedy WER:   0%|          | 0/246 [00:00<?, ?it/s]

Final Validation WER (Greedy): 0.5212

Sample predictions:

[1] REF : من وظفك
    HYP : من وظفك
--------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
--------------------------------------------------------------------------------
[3] REF : واحد
    HYP : واد
--------------------------------------------------------------------------------
[4] REF : متي سترجع
    HYP : متي ستعاجه
--------------------------------------------------------------------------------
[5] REF : ثمانية
    HYP : ثمانية
--------------------------------------------------------------------------------
[6] REF : ذهبت الي بيتي
    HYP : ذهبت لي بيتي
--------------------------------------------------------------------------------
[7] REF : ستة
    HYP : ستة
--------------------------------------------------------------------------------
[8] REF : كنا هنا
    HYP : كنا هنا
--------------------------------------------------------------------------------
[9] REF : م

Beam WER (beam=20):   0%|          | 0/246 [00:00<?, ?it/s]

Final Validation WER (Beam Search, beam_width=20): 0.5125
[1] REF : من وظفك
    HYP : من وظفك
--------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
--------------------------------------------------------------------------------
[3] REF : واحد
    HYP : واد
--------------------------------------------------------------------------------
[4] REF : متي سترجع
    HYP : متي ستعاجه
--------------------------------------------------------------------------------
[5] REF : ثمانية
    HYP : ثمانية
--------------------------------------------------------------------------------
[6] REF : ذهبت الي بيتي
    HYP : ذهبت لي بيتي
--------------------------------------------------------------------------------
[7] REF : ستة
    HYP : ستة
--------------------------------------------------------------------------------
[8] REF : كنا هنا
    HYP : كنا هنا
--------------------------------------------------------------------------------
[9] REF : مرث

NGram Rescore WER (beam=20, lm_w=0.6):   0%|          | 0/246 [00:00<?, ?it/s]

Final Validation WER (Manual Beam + N-gram Rescoring): 0.4850
[1] REF : من وظفك
    HYP : من وظفك
------------------------------------------------------------------------------------------
[2] REF : حسنا
    HYP : حسنا
------------------------------------------------------------------------------------------
[3] REF : واحد
    HYP : واحد
------------------------------------------------------------------------------------------
[4] REF : متي سترجع
    HYP : متي ستعاجه
------------------------------------------------------------------------------------------
[5] REF : ثمانية
    HYP : ثمانية
------------------------------------------------------------------------------------------
[6] REF : ذهبت الي بيتي
    HYP : ذهبت ليابيتي
------------------------------------------------------------------------------------------
[7] REF : ستة
    HYP : ستة
------------------------------------------------------------------------------------------
[8] REF : كنا هنا
    HYP : كنا هنا
-------------------

In [ ]:
# =========================================================
# 17) SAVE FINAL ZIP
# =========================================================
import zipfile
from datetime import datetime

OUTPUT_DIR = os.path.join(WORK_BASE, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_PATH = os.path.join(OUTPUT_DIR, f"asr_final_bigru_bundle_{timestamp}.zip")

results_summary = {
    "greedy_wer": float(greedy_wer),
    "beam_wer": float(beam_wer),
    "ngram_wer": float(ngram_wer),
    "best_val_loss": float(best_val_loss),
    "batch_size": BATCH_SIZE,
    "proj_dim": PROJ_DIM,
    "rnn_hidden": RNN_HIDDEN,
    "rnn_layers": RNN_LAYERS,
    "dropout": DROPOUT,
}

summary_json_path = os.path.join(OUTPUT_DIR, "results_summary.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

def save_prediction_txt(path, refs, hyps):
    with open(path, "w", encoding="utf-8") as f:
        n = min(len(refs), len(hyps))
        for i in range(n):
            f.write(f"[{i+1}] REF : {refs[i]}\n")
            f.write(f"    HYP : {hyps[i]}\n")
            f.write("-" * 80 + "\n")

greedy_txt = os.path.join(OUTPUT_DIR, "greedy_samples.txt")
beam_txt   = os.path.join(OUTPUT_DIR, "beam_samples.txt")
ngram_txt  = os.path.join(OUTPUT_DIR, "ngram_samples.txt")

save_prediction_txt(greedy_txt, greedy_refs, greedy_hyps)
save_prediction_txt(beam_txt, beam_refs, beam_hyps)
save_prediction_txt(ngram_txt, ngram_refs, ngram_hyps)

files_to_zip = [
    last_ckpt_path,
    best_ckpt_path,
    history_path,
    training_state_path,
    train_mel_csv,
    val_mel_csv,
    test_mel_csv,
    char2idx_path,
    idx2char_path,
    summary_json_path,
    greedy_txt,
    beam_txt,
    ngram_txt,
]

with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in files_to_zip:
        if os.path.exists(file_path):
            zf.write(file_path, arcname=os.path.basename(file_path))
            print("Added:", os.path.basename(file_path))
        else:
            print("Missing:", file_path)

print("\nZIP created successfully:")
print(ZIP_PATH)
print("\nOutput folder contents:")
print(os.listdir(OUTPUT_DIR))